# Panzer — Endpoints publicos

Este notebook muestra como usar `BinancePublicClient` para obtener datos
de mercado de Binance **sin necesidad de API keys**.

Panzer gestiona automaticamente:
- **Rate limiting** sincronizado con Binance (nunca te banean).
- **Calculo de pesos** por endpoint y parametros.
- **Sincronizacion de reloj** con el servidor.

Mercados soportados: `spot`, `um` (USDT-M Futures), `cm` (COIN-M Futures).

## 1. Crear un cliente

In [1]:
from panzer import BinancePublicClient

# Al crear el cliente se descarga /exchangeInfo para obtener los limites de peso.
# safety_ratio=0.9 significa que dormira al llegar al 90% del limite (ej: 5400/6000).
client = BinancePublicClient(market="spot", safety_ratio=0.9)

print(f"Mercado:     {client.market}")
print(f"Base URL:    {client.base_url}")
print(f"Max peso/min: {client.limiter.max_per_minute}")

2026-03-06 13:00:43     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-06 13:00:43     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90


Mercado:     spot
Base URL:    https://api.binance.com
Max peso/min: 6000


## 2. Sincronizar reloj

Antes de hacer muchas peticiones es recomendable sincronizar el reloj local
con el del servidor. Esto mejora la precision del rate limiter.

In [2]:
# Llama a /time 3 veces para estimar el desfase local vs servidor
client.ensure_time_offset_ready(min_samples=3)

offset_ms = client.time_offset.current_offset() * 1000
print(f"Desfase estimado: {offset_ms:.1f} ms")
print(f"Hora servidor (estimada): {client.now_server_ms()} ms")

Desfase estimado: -110.7 ms
Hora servidor (estimada): 1772798444846 ms


## 3. Klines (velas)

Datos OHLCV. Cada vela es una lista de 12 elementos:
`[open_time, open, high, low, close, volume, close_time, quote_vol, trades, taker_buy_base, taker_buy_quote, ignore]`

In [3]:
klines = client.klines("BTCUSDT", "1h", limit=5)

print(f"Obtenidas {len(klines)} velas de 1h para BTCUSDT\n")
for k in klines:
    open_price, high, low, close = k[1], k[2], k[3], k[4]
    volume = k[5]
    trades = k[8]
    print(f"  O={open_price:>12s}  H={high:>12s}  L={low:>12s}  C={close:>12s}  Vol={volume:>14s}  Trades={trades}")

Obtenidas 5 velas de 1h para BTCUSDT

  O=71056.44000000  H=71067.58000000  L=70735.85000000  C=70789.49000000  Vol=  569.42569000  Trades=159042
  O=70789.49000000  H=70808.65000000  L=70300.00000000  C=70565.56000000  Vol=  637.97560000  Trades=199817
  O=70565.56000000  H=70800.00000000  L=70467.88000000  C=70527.93000000  Vol=  469.81877000  Trades=123402
  O=70527.93000000  H=70643.27000000  L=70184.01000000  C=70212.95000000  Vol=  763.93539000  Trades=123500
  O=70212.94000000  H=70212.95000000  L=70182.31000000  C=70186.36000000  Vol=   37.20222000  Trades=2488


### Intervalos disponibles

`1s`, `1m`, `3m`, `5m`, `15m`, `30m`, `1h`, `2h`, `4h`, `6h`, `8h`, `12h`, `1d`, `3d`, `1w`, `1M`

In [4]:
# Klines con rango temporal (timestamps en milisegundos)
import time

end_ms = int(time.time() * 1000)
start_ms = end_ms - 24 * 60 * 60 * 1000  # ultimas 24h

klines_24h = client.klines("ETHUSDT", "15m", start_time=start_ms, end_time=end_ms, limit=1000)
print(f"Velas de 15m en las ultimas 24h: {len(klines_24h)}")

Velas de 15m en las ultimas 24h: 96


## 4. Trades recientes

In [5]:
trades = client.trades("BTCUSDT", limit=5)

print(f"Ultimos {len(trades)} trades de BTCUSDT:\n")
for t in trades:
    side = "SELL" if t["isBuyerMaker"] else "BUY "
    print(f"  id={t['id']}  {side}  price={t['price']:>12s}  qty={t['qty']:>12s}")

Ultimos 5 trades de BTCUSDT:

  id=6072513461  SELL  price=70186.36000000  qty=  0.00300000
  id=6072513462  BUY   price=70186.37000000  qty=  0.00016000
  id=6072513463  BUY   price=70186.37000000  qty=  0.00125000
  id=6072513464  BUY   price=70186.37000000  qty=  0.00500000
  id=6072513465  SELL  price=70186.36000000  qty=  0.00214000


## 5. Trades agregados (aggTrades)

Los aggTrades comprimen varios trades atomicos en uno cuando comparten precio y lado.
Cada aggTrade tiene un rango de trade IDs (`f` a `l`).

In [6]:
agg = client.agg_trades("BTCUSDT", limit=5)

print(f"Ultimos {len(agg)} aggTrades:\n")
for t in agg:
    maker = "maker" if t["m"] else "taker"
    atomic_count = t["l"] - t["f"] + 1
    print(f"  aggId={t['a']}  price={t['p']:>12s}  qty={t['q']:>10s}  {maker}  ({atomic_count} trades atomicos)")

Ultimos 5 aggTrades:

  aggId=3894699994  price=70185.29000000  qty=0.00300000  maker  (1 trades atomicos)
  aggId=3894699995  price=70185.29000000  qty=0.00300000  maker  (1 trades atomicos)
  aggId=3894699996  price=70185.29000000  qty=0.00300000  maker  (1 trades atomicos)
  aggId=3894699997  price=70185.29000000  qty=0.00300000  maker  (1 trades atomicos)
  aggId=3894699998  price=70185.30000000  qty=0.00149000  taker  (1 trades atomicos)


## 6. Order book (depth)

In [7]:
book = client.depth("BTCUSDT", limit=5)

best_bid = float(book["bids"][0][0])
best_ask = float(book["asks"][0][0])
spread = best_ask - best_bid

print(f"Order book BTCUSDT (top 5 niveles)\n")
print(f"  {'BIDS (compra)':>30s}  |  {'ASKS (venta)':<30s}")
print(f"  {'price':>14s} {'qty':>14s}  |  {'price':<14s} {'qty':<14s}")
print(f"  {'-'*30}  |  {'-'*30}")
for i in range(5):
    bid_p, bid_q = book["bids"][i]
    ask_p, ask_q = book["asks"][i]
    print(f"  {bid_p:>14s} {bid_q:>14s}  |  {ask_p:<14s} {ask_q:<14s}")

print(f"\n  Best bid: {best_bid}  |  Best ask: {best_ask}  |  Spread: {spread:.2f}")

Order book BTCUSDT (top 5 niveles)

                   BIDS (compra)  |  ASKS (venta)                  
           price            qty  |  price          qty           
  ------------------------------  |  ------------------------------
  70183.35000000     0.30388000  |  70183.36000000 4.36732000    
  70183.34000000     0.00056000  |  70183.37000000 0.02434000    
  70182.52000000     0.00017000  |  70183.38000000 0.05241000    
  70182.37000000     0.00008000  |  70183.52000000 0.00200000    
  70182.36000000     0.00029000  |  70184.62000000 0.00008000    

  Best bid: 70183.35  |  Best ask: 70183.36  |  Spread: 0.01


## 7. Exchange info

Metadata del exchange: simbolos disponibles, filtros de precio/cantidad, etc.

In [8]:
# Info de un simbolo concreto
info = client.exchange_info(symbol="BTCUSDT")
sym = info["symbols"][0]

print(f"Simbolo:   {sym['symbol']}")
print(f"Status:    {sym['status']}")
print(f"Base:      {sym['baseAsset']}")
print(f"Quote:     {sym['quoteAsset']}")
print(f"Filtros:   {len(sym['filters'])}")
for f in sym["filters"]:
    print(f"  - {f['filterType']}")

Simbolo:   BTCUSDT
Status:    TRADING
Base:      BTC
Quote:     USDT
Filtros:   11
  - PRICE_FILTER
  - LOT_SIZE
  - ICEBERG_PARTS
  - MARKET_LOT_SIZE
  - TRAILING_DELTA
  - PERCENT_PRICE_BY_SIDE
  - NOTIONAL
  - MAX_NUM_ORDERS
  - MAX_NUM_ORDER_LISTS
  - MAX_NUM_ALGO_ORDERS
  - MAX_NUM_ORDER_AMENDS


## 8. Multiples mercados

El mismo API funciona para Spot, USDT-M Futures y COIN-M Futures.
Solo cambia el parametro `market`.

In [9]:
# Comparar precio de BTC en los 3 mercados
for market in ["spot", "um", "cm"]:
    c = BinancePublicClient(market=market)
    symbol = "BTCUSDT" if market != "cm" else "BTCUSD_PERP"
    kline = c.klines(symbol, "1m", limit=1)
    close = kline[0][4]
    print(f"  {market:>4s}  {symbol:<16s}  close={close}")

2026-03-06 13:00:48     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-06 13:00:49     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90
2026-03-06 13:00:49     INFO [panzer.binance.public.um] Inicializando BinancePublicClient(market=um)


  spot  BTCUSDT           close=70184.13000000


2026-03-06 13:00:49     INFO [panzer.binance.public.um] Rate limiter inicializado: max_per_minute=2400 safety_ratio=0.90
2026-03-06 13:00:50     INFO [panzer.binance.public.cm] Inicializando BinancePublicClient(market=cm)


    um  BTCUSDT           close=70157.40


2026-03-06 13:00:50     INFO [panzer.binance.public.cm] Rate limiter inicializado: max_per_minute=2400 safety_ratio=0.90


    cm  BTCUSD_PERP       close=70161.0


## 9. Endpoint generico con `get()`

Para cualquier endpoint publico que no tenga wrapper, puedes usar `get()` directamente.
El peso se calcula automaticamente o puedes pasarlo manualmente.

In [10]:
# Ticker 24h — no tiene wrapper dedicado
ticker = client.get("/api/v3/ticker/24hr", params={"symbol": "BTCUSDT"})

print(f"BTCUSDT 24h:")
print(f"  Precio actual:   {ticker['lastPrice']}")
print(f"  Cambio 24h:      {ticker['priceChangePercent']}%")
print(f"  Volumen 24h:     {ticker['volume']} BTC")
print(f"  Max 24h:         {ticker['highPrice']}")
print(f"  Min 24h:         {ticker['lowPrice']}")

BTCUSDT 24h:
  Precio actual:   70193.76000000
  Cambio 24h:      -3.735%
  Volumen 24h:     21441.91986000 BTC
  Max 24h:         73063.38000000
  Min 24h:         70143.19000000


## 10. Rate limiter — inspeccion en tiempo real

Puedes consultar el estado del limiter en cualquier momento para ver
cuanto peso has consumido en la ventana actual.

In [11]:
lim = client.limiter

print(f"Peso usado (local):      {lim.used_local}")
print(f"Peso usado (servidor):   {lim.last_server_used}")
print(f"Limite por minuto:       {lim.max_per_minute}")
print(f"Ratio de seguridad:      {lim.safety_ratio}")
print(f"Limite efectivo:         {int(lim.max_per_minute * lim.safety_ratio)}")
print(f"Margen restante:         {int(lim.max_per_minute * lim.safety_ratio) - lim.used_local}")

Peso usado (local):      105
Peso usado (servidor):   105
Limite por minuto:       6000
Ratio de seguridad:      0.9
Limite efectivo:         5400
Margen restante:         5295


## 11. Manejo de errores

In [12]:
from panzer.errors import BinanceAPIException

try:
    client.klines("SIMBOLO_INVALIDO", "1m")
except BinanceAPIException as e:
    print(f"Error capturado:")
    print(f"  HTTP status:  {e.status_code}")
    print(f"  Binance code: {e.error_payload.code}")
    print(f"  Mensaje:      {e.error_payload.msg}")
    print(f"  URL:          {e.url}")

2026-03-06 13:00:52    ERROR [panzer.errors] BinanceAPIException raised: status=400 method=GET url=https://api.binance.com/api/v3/klines?symbol=SIMBOLO_INVALIDO&interval=1m&limit=500 code=-1121 msg=Invalid symbol.


Error capturado:
  HTTP status:  400
  Binance code: -1121
  Mensaje:      Invalid symbol.
  URL:          https://api.binance.com/api/v3/klines?symbol=SIMBOLO_INVALIDO&interval=1m&limit=500
